<a href="https://colab.research.google.com/github/dhanalakshmikeerthi/Home-Bakery/blob/main/Day_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import functools

def log_call(func):                              # takes a function…
    @functools.wraps(func)                       # (keeps the name and docstring)
    def wrapper(*args, **kwargs):
        print(f"→ calling {func.__name__}{args}")
        result = func(*args, **kwargs)           # run the original
        print(f"← {func.__name__} returned {result!r}")
        return result
    return wrapper                               # …returns a wrapped one

@log_call                                        # same as: get_price = log_call(get_price)
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

In [11]:
!pip install -q openai                           # once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("Groq-api_key"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",             # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Why don't scientists trust atoms?

Because they make up everything!


In [15]:
from openai import RateLimitError, AuthenticationError, APIConnectionError

try:
  history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
  ]
  response = client.chat.completions.create(model="openai/gpt-oss-20b", messages=history)
  reply = response.choices[0].message.content
except RateLimitError:
  reply = "We're a little busy. Please try again in a minute."
except AuthenticationError:
  reply = "Setup problem: please tell the owner the API key isn't working."
except APIConnectionError:
  reply = "I can't reach the server right now. Please try again shortly."
print(reply)

Hello! 👋 How can I help you today?


In [16]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-20b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply

print("Total for 6 cupcakes:", get_price("cupcake", 6))
while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

   [log] get_price('cupcake', 6) -> 480
Total for 6 cupcakes: 480
You: Can u give me order by this evening 
Bot: I’m sorry, we need 24‑hour notice for orders. Could you let me know what you’d like to order and your name?
You: What is cost of plum cake
Bot: The plum cake costs ₹500. Let me know if you’d like to place an order—just a heads‑up, we need 24‑hour notice.
You: Will the taste is good
Bot: Absolutely—our plum cake is a crowd‑pleaser and we offer an eggless version too. Let me know your name and if you’d like to place an order (just a reminder we need 24‑hour notice).
You: Yes please
Bot: Sure! I’ve noted you’d like a plum cake. May I have your name, please? Just a reminder, we need 24‑hour notice for orders. Once I have your details, I’ll confirm the order and price (₹500).
You: My name is Dhana 
Bot: Hi Dhana, your plum cake (₹500) is confirmed. We need 24‑hour notice before delivery. Let me know if you’d like to proceed or have any questions. Thanks!
You: Ok thank u 
Bot: Tha